In [ ]:
import os
import shutil
import pandas as pd
import cv2

def copy_videos_and_calculate_fps(excel_path, source_folder, destination_folder):
    # Read the Excel sheet
    df = pd.read_excel(excel_path)

    # Iterate over each row in the Excel sheet
    for index, row in df.iterrows():
        genre = str(row['genre'])
        genre_sub = str(row['genre_sub'])
        video_id = str(row['video_id'])

        # Construct the source and destination paths
        source_video_path = os.path.join(source_folder, genre, genre_sub, video_id, f"{video_id}.mp4")
        destination_video_path = os.path.join(destination_folder, f"{video_id}.mp4")

        # Check if the source video file exists
        if os.path.exists(source_video_path):
            # Copy the video file to the destination folder
            shutil.copy(source_video_path, destination_video_path)

            # Calculate the video FPS
            cap = cv2.VideoCapture(destination_video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            cap.release()

            # Update the 'og_fps' column in the Excel sheet
            df.at[index, 'og_fps'] = fps

            print(f"Video {video_id} copied to {destination_folder}. FPS: {fps}")
        else:
            print(f"Warning: Video file not found for video {video_id}.")

    # Save the updated Excel sheet
    df.to_excel(excel_path, index=False)

# Replace these paths with your actual paths

# Call the function
copy_videos_and_calculate_fps(excel_path, source_folder, destination_folder)


In [ ]:
import pandas as pd

def generate_label(trimming_sec, og_duration):
    label = ['0'] * og_duration  # Initialize label with '0' for the entire duration

    # Split trimming_sec into individual trimming ranges
    trimming_ranges = [tuple(map(int, trim.split(' to '))) for trim in trimming_sec.split(', ')]

    # Iterate over trimming ranges and update the label
    for start, end in trimming_ranges:
        label[start - 1:end] = ['1'] * (end - start + 1)

    return ''.join(label)

def generate_label_og_fps(label_1fps, og_fps):
    label_og_fps = ''
    
    for char in label_1fps:
        label_og_fps += '0' * int(og_fps) if char == '0' else '1' * int(og_fps)

    return label_og_fps

def process_excel_sheet(excel_path):
    # Read the Excel sheet
    df = pd.read_excel(excel_path)

    # Apply the function to create the 'label_1fps' column
    df['label_1fps'] = df.apply(lambda row: generate_label(row['trimming_sec'], row['og_duration']), axis=1)

    # Apply the function to create the 'label_og_fps' column
    df['label_og_fps'] = df.apply(lambda row: generate_label_og_fps(row['label_1fps'], row['og_fps']), axis=1)

    # Save the updated Excel sheet
    df.to_excel(excel_path, index=False)

# Replace this path with your actual path

# Call the function
process_excel_sheet(excel_path)


In [ ]:
import os
import cv2
import pandas as pd

# Replace with the actual file path

# Replace with the actual folder path


# Load your Excel sheet into a DataFrame
try:
    df = pd.read_excel(excel_file_path, sheet_name="Sheet1")  # Replace "Sheet1" with the actual sheet name
except Exception as e:
    print(f"Error reading Excel file: {e}")
    raise

# Add 'n_frames' column if not exists
if 'n_frames' not in df.columns:
    df['n_frames'] = None

# Iterate through rows in the DataFrame
for index, row in df.iterrows():
    video_id = str(row['video_id'])  # Assuming 'video_id' is the column name in your DataFrame
    video_path = os.path.join(video_folder, f'{video_id}.mp4')

    # Check if the video file exists
    if os.path.exists(video_path):
        # Open the video file
        cap = cv2.VideoCapture(video_path)

        # Get the number of frames
        num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Update 'n_frames' column
        df.loc[index, 'n_frames'] = num_frames

        # Release the video capture object
        cap.release()

        # Check 'label_og_fps' and update if necessary
        label_og_fps = str(row['label_og_fps'])
        while len(label_og_fps) < num_frames:
            label_og_fps += '0'
        df.loc[index, 'label_og_fps'] = label_og_fps
        while len(label_og_fps) > num_frames:
            label_og_fps = label_og_fps[-num_frames:]
        df.loc[index, 'label_og_fps'] = label_og_fps

        # Additional print statements for debugging
        print(f"Video ID: {video_id}, Original Length: {len(str(row['label_og_fps']))}, Num Frames: {num_frames}, Updated Length: {len(label_og_fps)}")

    else:
        print(f"Video file not found: {video_path}")

# Save the updated DataFrame to a new Excel file

df.to_excel(output_excel_path, index=False)  # Replace with the desired file path
print(f"Updated DataFrame saved to: {output_excel_path}")


In [ ]:
import pandas as pd
import json
import os

class LongStringDecoder(json.JSONDecoder):
    def decode(self, s, _w=json.decoder.WHITESPACE.match):
        result = super().decode(s, _w)
        return result

def convert_to_json(excel_path, output_folder):
    # Read the Excel sheet
    df = pd.read_excel(excel_path)

    # Iterate over rows in the DataFrame
    for _, row in df.iterrows():
        video_id = str(row['video_id'])

        # Convert 'label_og_fps' column to a list of integers
        label_og_fps_str = row['label_og_fps']
        parsed_data = [int(char) for char in label_og_fps_str]

        # Create a dictionary for the JSON format
        json_data = {"user_summary": [parsed_data]}

        # Convert to JSON and save to file
        json_filename = f"{video_id}.json"
        json_filepath = os.path.join(output_folder, json_filename)
        with open(json_filepath, 'w') as json_file:
            json.dump(json_data, json_file)

# Replace these paths with your actual paths


# Call the function
convert_to_json(excel_path, output_folder)


In [ ]:
###########################################################################################################

In [ ]:
cd DSNet

In [ ]:
cd src

In [ ]:
import os

# Set the environment variable
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Now run your command
!python make_dataset.py --video-dir "" --label-dir "" --save-path "C:\Users\Desktop\dsnet\custom_dataset.h5" --sample-rate 15


In [ ]:
##################################################################################################################

In [ ]:
!python make_split.py --dataset "C:\Users\Desktop\dsnet\custom_dataset.h5" --train-ratio 0.67 --save-path "C:\Users\dsnet\custom_dataset.yml"

In [ ]:
# from ortools.algorithms import pywrapknapsack_solver

!python train.py anchor-based --model-dir "C:\Users\Desktop\dsnet\custom_dataset" --splits "C:\Users\Desktop\dsnet\custom_dataset.yml"
!python evaluate.py anchor-based --model-dir "C:\Users\Desktop\dsnet\custom_dataset" --splits "C:\Users\Desktop\dsnet\custom_dataset.yml"